# 15.1 语言能力评估 (Language Evaluation)

评估大语言模型的语言理解和生成能力，是模型开发中最重要的环节之一。

## 评估体系总览

| 评估维度 | 核心基准 | 评估指标 | 能力层次 |
|----------|----------|----------|----------|
| 语言建模 | WikiText/Pile | PPL | 基础 |
| 知识理解 | MMLU | Accuracy | 知识 |
| 常识推理 | HellaSwag/ARC | Accuracy | 推理 |
| 生成质量 | MT-Bench | Score | 生成 |
| 代码能力 | HumanEval | pass@k | 编程 |
| 长文本 | LongBench | F1/EM | 长上下文 |

## 1. 语言建模指标

语言建模指标衡量模型对文本的预测能力，是最基础的评估。

### 困惑度（Perplexity）
- PPL = exp(交叉熵损失)
- PPL越低，模型预测越准确
- 在不同领域数据上计算PPL可以评估泛化能力

### Bits Per Character (BPC)
- 每个字符的平均比特数
- BPC = 交叉熵 / ln(2)
- 更细粒度的语言建模指标

### 关键注意事项
- PPL依赖分词器，不同分词器的PPL不可直接比较
- 评估时使用因果掩码（不能看到未来token）
- 报告PPL时应注明评估数据集和分词器

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class PerplexityEvaluator:
    def __init__(self, model, tokenizer=None):
        self.model = model
        self.tokenizer = tokenizer

    def compute_ppl(self, logits, targets, ignore_index=-100):
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               targets.reshape(-1), ignore_index=ignore_index)
        ppl = math.exp(min(loss.item(), 20))
        return ppl, loss.item()

    def compute_ppl_by_chunk(self, logits, targets, chunk_size=512):
        ppls = []
        for i in range(0, logits.shape[1], chunk_size):
            end = min(i + chunk_size, logits.shape[1])
            chunk_logits = logits[:, i:end, :]
            chunk_targets = targets[:, i:end]
            ppl, _ = self.compute_ppl(chunk_logits, chunk_targets)
            ppls.append(ppl)
        return ppls

    def compute_bpc(self, logits, targets, chars_per_token=4.0):
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               targets.reshape(-1))
        bpc = loss.item() / math.log(2) / chars_per_token
        return bpc

class SimpleLM(nn.Module):
    def __init__(self, vocab_size=500, d_model=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 2, d_model * 4, batch_first=True), 2
        )
        self.head = nn.Linear(d_model, vocab_size)
    def forward(self, x):
        return self.head(self.transformer(self.embed(x)))

model = SimpleLM()
evaluator = PerplexityEvaluator(model)

logits = torch.randn(4, 100, 500)
targets = torch.randint(0, 500, (4, 100))

ppl, ce_loss = evaluator.compute_ppl(logits, targets)
bpc = evaluator.compute_bpc(logits, targets)
chunk_ppls = evaluator.compute_ppl_by_chunk(logits, targets, chunk_size=25)

print('=== Language Modeling Metrics ===')
print(f'Cross-entropy loss: {ce_loss:.4f}')
print(f'Perplexity: {ppl:.2f}')
print(f'Bits per character: {bpc:.4f}')
print(f'\nPPL by chunk (chunk_size=25):')
for i, p in enumerate(chunk_ppls):
    print(f'  Chunk {i+1}: PPL={p:.2f}')

print(f'\nKey: PPL is the primary metric for language modeling quality.')
print(f'Lower PPL = better prediction. Compare only with same tokenizer.')
print(f'Chunk-wise PPL reveals position-dependent performance.')

## 2. 理解任务评估

理解任务评估衡量模型的知识广度和推理能力。

### 核心基准

**MMLU（Massive Multitask Language Understanding）**
- 57个学科的多选题
- 涵盖STEM、人文、社科等
- 5-shot评估，测试知识广度
- 当前SOTA：~90%（GPT-4级别）

**HellaSwag**
- 常识推理完形填空
- 选择最合理的句子续写
- 测试常识推理能力

**ARC（AI2 Reasoning Challenge）**
- ARC-Easy：简单科学题
- ARC-Challenge：困难科学题
- 测试科学推理能力

### 评估方法
- **生成式**：模型生成答案，与标准答案比较
- **判别式**：计算各选项的概率，选概率最高的
- 判别式更稳定，推荐使用

In [ ]:
class UnderstandingEvaluator:
    def __init__(self, model=None):
        self.model = model

    def discriminative_eval(self, option_logits):
        preds = option_logits.argmax(dim=-1)
        return preds

    def generative_eval(self, generated_ids, reference_ids):
        exact_match = (generated_ids == reference_ids).float().mean()
        return exact_match

    def evaluate_mmlu_style(self, model, questions, options, answers, n_shots=5):
        correct = 0
        total = len(answers)
        for i, (q, opts, ans) in enumerate(zip(questions, options, answers)):
            option_probs = F.softmax(torch.randn(len(opts)), dim=-1)
            pred = option_probs.argmax().item()
            if pred == ans:
                correct += 1
        return correct / total

    def few_shot_prompt_builder(self, question, examples, n_shots=5):
        prompt_parts = []
        for ex in examples[:n_shots]:
            prompt_parts.append(f'Q: {ex["question"]}')
            for j, opt in enumerate(ex['options']):
                prompt_parts.append(f'  {chr(65+j)}. {opt}')
            prompt_parts.append(f'A: {chr(65+ex["answer"])}')
        prompt_parts.append(f'Q: {question}')
        return '\n'.join(prompt_parts)

evaluator = UnderstandingEvaluator()

option_logits = torch.randn(8, 4)
preds = evaluator.discriminative_eval(option_logits)

questions = ['What is 2+2?', 'Capital of France?']
options = [['3', '4', '5', '6'], ['London', 'Paris', 'Berlin', 'Madrid']]
answers = [1, 1]
acc = evaluator.evaluate_mmlu_style(None, questions, options, answers)

print('=== Understanding Task Evaluation ===')
print(f'\nDiscriminative evaluation:')
print(f'  Option logits shape: {option_logits.shape}')
print(f'  Predictions: {preds.tolist()}')
print(f'\nMMLU-style evaluation:')
print(f'  Accuracy: {acc:.2%}')

examples = [
    {'question': 'What is 1+1?', 'options': ['1', '2', '3', '4'], 'answer': 1},
    {'question': 'What is 3+3?', 'options': ['5', '6', '7', '8'], 'answer': 1},
]
prompt = evaluator.few_shot_prompt_builder('What is 2+2?', examples, n_shots=2)
print(f'\nFew-shot prompt example:')
print(prompt)

print(f'\nKey: Discriminative evaluation is more stable than generative.')
print(f'MMLU is the most important benchmark for knowledge breadth.')
print(f'Few-shot prompting provides task context without fine-tuning.')

## 3. 生成任务评估

生成任务评估衡量模型的输出质量，包括自动指标和人工评估。

### 自动评估指标

| 指标 | 适用任务 | 评估维度 | 局限 |
|------|----------|----------|------|
| BLEU | 翻译 | n-gram精确率 | 忽略语义 |
| ROUGE | 摘要 | n-gram召回率 | 忽略语义 |
| BERTScore | 通用 | 语义相似度 | 需要BERT模型 |
| METEOR | 翻译 | 同义词+词序 | 计算复杂 |
| CIDEr | 图像描述 | TF-IDF加权 | 特定领域 |

### LLM-as-Judge
用强模型（GPT-4）评估弱模型的输出质量，适用于开放性任务。
- **MT-Bench**：8个类别的多轮对话评估
- **AlpacaEval**：指令遵循评估
- **Arena Hard**：竞技场式对比评估

In [ ]:
class GenerationEvaluator:
    def __init__(self):
        pass

    def compute_bleu(self, reference, hypothesis, max_n=4):
        ref_tokens = reference.split()
        hyp_tokens = hypothesis.split()
        precisions = []
        for n in range(1, max_n + 1):
            ref_ngrams = [tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1)]
            hyp_ngrams = [tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens)-n+1)]
            if not hyp_ngrams:
                precisions.append(0)
                continue
            matches = sum(1 for ng in hyp_ngrams if ng in ref_ngrams)
            precisions.append(matches / len(hyp_ngrams))
        avg_precision = sum(precisions) / len(precisions) if precisions else 0
        bp = min(1, math.exp(1 - len(ref_tokens) / max(1, len(hyp_tokens))))
        return bp * avg_precision

    def compute_rouge_l(self, reference, hypothesis):
        ref_tokens = reference.split()
        hyp_tokens = hypothesis.split()
        m = len(ref_tokens)
        n = len(hyp_tokens)
        if m == 0 or n == 0:
            return 0
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if ref_tokens[i-1] == hyp_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        lcs = dp[m][n]
        precision = lcs / n
        recall = lcs / m
        if precision + recall == 0:
            return 0
        return 2 * precision * recall / (precision + recall)

    def compute_bertscore(self, ref_emb, hyp_emb):
        sim = F.cosine_similarity(ref_emb.unsqueeze(1), hyp_emb.unsqueeze(0), dim=-1)
        precision = sim.max(dim=0).values.mean()
        recall = sim.max(dim=1).values.mean()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return {'precision': precision.item(), 'recall': recall.item(), 'f1': f1.item()}

evaluator = GenerationEvaluator()

ref = 'the cat sat on the mat'
hyp_good = 'the cat sat on the mat'
hyp_ok = 'the cat is on the mat'
hyp_bad = 'a dog ran in the park'

print('=== Generation Evaluation ===')
print(f'Reference: "{ref}"')
print(f'\n{"Hypothesis":<30} {"BLEU":>8} {"ROUGE-L":>8}')
for name, hyp in [('Perfect', hyp_good), ('Partial', hyp_ok), ('No match', hyp_bad)]:
    bleu = evaluator.compute_bleu(ref, hyp)
    rouge = evaluator.compute_rouge_l(ref, hyp)
    print(f'{name:<30} {bleu:>8.4f} {rouge:>8.4f}')

ref_emb = F.normalize(torch.randn(6, 32), dim=-1)
hyp_emb = F.normalize(torch.randn(5, 32), dim=-1)
bertscore = evaluator.compute_bertscore(ref_emb, hyp_emb)
print(f'\nBERTScore: P={bertscore["precision"]:.3f}, R={bertscore["recall"]:.3f}, F1={bertscore["f1"]:.3f}')

print(f'\nKey: BLEU measures precision (translation), ROUGE measures recall (summarization).')
print(f'BERTScore captures semantic similarity, more robust to paraphrasing.')
print(f'No single metric is perfect. Use multiple metrics + human evaluation.')

## 4. 综合评估框架

工业级评估需要系统化的框架，覆盖多个维度。

### 评估维度
| 维度 | 基准 | 权重 | 说明 |
|------|------|------|------|
| 知识 | MMLU | 高 | 知识广度 |
| 推理 | GSM8K/MATH | 高 | 逻辑推理 |
| 代码 | HumanEval | 高 | 编程能力 |
| 语言 | HellaSwag/ARC | 中 | 语言理解 |
| 对话 | MT-Bench | 中 | 对话质量 |
| 安全 | TruthfulQA | 中 | 安全性 |
| 长文本 | LongBench | 低 | 长上下文 |

### 评估最佳实践
1. **多基准综合**：不要只看一个基准，综合多个基准的得分
2. **对比基线**：与同等规模的开源模型对比
3. **A/B测试**：真实用户场景的A/B测试最重要
4. **定期评估**：每次模型更新后重新评估
5. **评估数据去污染**：确保评估数据不在训练集中
6. **报告标准差**：多次运行取平均，报告标准差